# P08 — Konvolucija zvuka s impulsnim odzivom prostorije

Bilježnica uz predavanje P08 i **poglavlje 8 udžbenika *Signali i sustavi*** (Konvolucija i
impulsni odziv). Suhi zvučni zapis, snimljen bez odjeka, konvoluiramo sa snimkom pucnja u
prostoriji i čujemo kako bi isti zvuk zvučao u toj prostoriji. Umjesto zadanih zapisa možete
učitati vlastiti zvučni zapis i vlastiti impulsni odziv (odjeljak *Postavke* ispod).

Ćelije pokrenite redom (Shift + Enter) ili izbornikom *Run → Run All Cells*. Za zvuk trebaju
zvučnici ili slušalice. Bilježnica radi bez interneta.

In [ ]:
import html
import os
import warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from scipy.io import wavfile
from math import gcd
from scipy.signal import fftconvolve, resample_poly
from IPython.display import Audio, display
import ipywidgets as widgets

ZVUK = Path("zvuk")                    # zadani (zamjenski) zapisi
MOJ_ZVUK = Path("moj_zvuk")            # vlastite snimke (nije u gitu)
BOJA_X, BOJA_H, BOJA_Y = "#840B55", "#5B6770", "#222222"     # bordo, siva, tamna


def broj(v, decimala=2):
    """Broj s decimalnim zarezom (1,50)."""
    return f"{v:.{decimala}f}".replace(".", ",")


zarez = FuncFormatter(lambda v, _: f"{v:g}".replace(".", ","))   # decimalni zarez na osima
plt.rcParams.update({"axes.grid": True, "grid.alpha": 0.3})

## Postavke: zadani ili vlastiti zapisi

Bilježnica zadano koristi zamjenske zapise iz mape `zvuk/`. Za **vlastiti ulazni zapis**
(govor, glazba, instrument) ili **vlastiti impulsni odziv** (npr. prasak balona u dvorani)
upišite putanju između navodnika u ćeliji ispod, pa pokrenite sve ćelije (*Run → Run All
Cells*). Prazni navodnici `""` znače zadani zapis. Druga mogućnost je gumb za prijenos
datoteke u idućoj ćeliji.

Bilježnica čita samo **WAV** (mono ili stereo, 8, 16, 24 ili 32 bita, bilo koja frekvencija
uzorkovanja). Zapis iz mobitela (M4A, MP3, OGG, FLAC …) najprije pretvorite u WAV, npr.
besplatnim programom **Audacity**: *File → Open* (otvorite snimku), zatim *File → Export
Audio…* (u starijim inačicama *File → Export → Export as WAV*), vrsta datoteke *WAV*,
kodiranje *Signed 16-bit PCM*. Ako je instaliran program ffmpeg, isto radi naredba
`ffmpeg -i snimka.m4a snimka.wav`.

In [ ]:
# ===== POSTAVKE: vlastiti zapisi (mijenjajte samo ono između navodnika) =====
# Putanja je relativna prema mapi ove bilježnice. Prazno ("") = zadani zapis iz zvuk/.
ULAZ = ""        # ulazni („suhi”) zapis x[n], npr. "moj_zvuk/govor.wav"
ODZIV = ""       # impulsni odziv h[n], npr. "moj_zvuk/pucanj_dvorana.wav"
MAX_ULAZ_S = 15  # s: dulji ulaz skraćuje se na prvih MAX_ULAZ_S sekundi
REP_DB = -60     # dB: rep vlastitog odziva tiši od ove razine se odreže (None: bez rezanja)

# Za automatske testove: prazno polje preuzima vrijednost varijable okoline P08_ULAZ
# odnosno P08_ODZIV, ako je postavljena. Na predavanju to ne treba: upišite putanju gore.
ULAZ = ULAZ or os.environ.get("P08_ULAZ", "")
ODZIV = ODZIV or os.environ.get("P08_ODZIV", "")

In [ ]:
# Prijenos datoteke gumbom (JupyterLab; radi i bez interneta). Gumb otvori odabir datoteke.
# Prenesena datoteka sprema se u moj_zvuk/ i zamjenjuje ULAZ odnosno ODZIV iz ćelije iznad.
# Nakon prijenosa ponovno pokrenite sve ćelije (Run → Run All Cells): gumbi pamte datoteku
# do ponovnog pokretanja jezgre (Kernel → Restart Kernel) ili do naredbe gumb_ulaz.value = ()
# odnosno gumb_odziv.value = (). Bez prijenosa ćelija ništa ne čeka.
# Datoteke veće od oko 10 MB radije kopirajte u moj_zvuk/ ili ih mišem povucite u
# preglednik datoteka JupyterLaba (lijevi stupac), u mapu moj_zvuk/, i upišite putanju gore.


def spremi_prijenos(preneseno, mapa=MOJ_ZVUK):
    """Zadnja datoteka prenesena gumbom -> spremljena u mapu; vraća njezinu putanju ili ""."""
    if not preneseno:
        return ""
    datoteka = preneseno[-1]                   # ipywidgets 8: n-torka rječnika
    mapa = Path(mapa)
    mapa.mkdir(parents=True, exist_ok=True)
    put = mapa / Path(datoteka["name"]).name    # samo ime, bez mape iz preglednika
    put.write_bytes(bytes(datoteka["content"]))
    return put.as_posix()


if "gumb_ulaz" not in globals():                # isti gumbi i nakon Run All Cells
    gumb_ulaz = widgets.FileUpload(accept=".wav", description="ulaz x (WAV)")
    gumb_odziv = widgets.FileUpload(accept=".wav", description="odziv h (WAV)")
    stanje = widgets.HTML("nije prenesena nijedna datoteka")

    def _preneseno(_promjena):
        imena = [html.escape(g.value[-1]["name"]) for g in (gumb_ulaz, gumb_odziv) if g.value]
        stanje.value = f"preneseno: {', '.join(imena)}; sada <b>Run → Run All Cells</b>"

    gumb_ulaz.observe(_preneseno, "value")
    gumb_odziv.observe(_preneseno, "value")

display(widgets.HBox([gumb_ulaz, gumb_odziv, stanje]))
ULAZ = spremi_prijenos(gumb_ulaz.value) or ULAZ
ODZIV = spremi_prijenos(gumb_odziv.value) or ODZIV
print(f"ulaz:  {ULAZ or 'zadani zvuk/suhi_zvuk.wav'}")
print(f"odziv: {ODZIV or 'zadani zvuk/odziv_pucanj.wav'}")

### Kako snimiti vlastiti impulsni odziv

1. U dvorani, učionici, hodniku ili stubištu napravite kratak i jak zvuk: prasak balona,
   glasan pljesak dlanovima ili startni pištolj (samo uz dopuštenje i na siguran način).
   Takav zvuk je približno delta, pa je njegova snimka impulsni odziv prostorije.
2. Mobitel ili mikrofon postavite nekoliko metara od izvora (npr. 3 do 5 m). Za vrijeme
   snimanja u prostoriji mora biti tiho.
3. Pokrenite snimanje, pričekajte oko pola sekunde, napravite prasak i snimajte još 2 do
   3 s dok odjek ne utihne. Vrh praska ne smije biti odrezan (izobličen): ako jest, odmaknite
   mobitel ili smanjite osjetljivost mikrofona.
4. Snimku pretvorite u WAV (vidi gore) i spremite je u mapu `moj_zvuk/`, npr. kao
   `moj_zvuk/pucanj_dvorana.wav`. U postavkama upišite `ODZIV = "moj_zvuk/pucanj_dvorana.wav"`.

Tišinu prije praska i tihi rep na kraju bilježnica odreže sama (ispiše koliko). Vlastiti
ulazni zapis snimite blizu mikrofona i u prigušenom prostoru (npr. u automobilu ili u
ormaru među odjećom), inače u njemu već ima odjeka prostorije.

**Licenca.** Mapa `moj_zvuk/` nije u gitu, pa vlastite snimke ostaju samo na vašem računalu.
U repozitorij (mapa za zapise uz bilježnicu, s izvorom i licencom u `LICENSE-sadrzaj.md`)
snimka smije samo ako ima licencu koja dopušta objavu (CC0, CC BY 4.0 ili CC BY-SA 4.0) i
ako se na njoj ne čuju drugi ljudi bez njihova pristanka. Tuđe snimke i snimke s interneta
bez takve licence ostaju u `moj_zvuk/`.

## 1. Konvolucija i impulsni odziv

**Impulsni odziv** $h(t)$ linearnog vremenski nepromjenjivog (LTI) sustava je njegov odziv na
Diracovu deltu $\delta(t)$. Odziv na bilo koji ulaz $x(t)$ je **konvolucija** ulaza s
impulsnim odzivom:

$$y(t) = (x*h)(t) = \int_{-\infty}^{\infty} x(\tau)\,h(t-\tau)\,d\tau .$$

Računalo radi s uzorcima $x[n]=x(nT)$, gdje je $T=1/f_s$ razmak uzoraka. Integral tada
postaje zbroj, **konvolucijska suma** (poglavlje 12), do faktora $T$ koji ionako uklanja
normiranje glasnoće:

$$y[n] = \sum_{k} x[k]\,h[n-k] .$$

**Prostorija kao sustav.** Dok izvor zvuka i mikrofon stoje na istim mjestima, put zvuka
kroz zrak i odbijanje od zidova približno su linearni i vremenski nepromjenjivi. Pucanj (ili
prasak balona) je vrlo kratak i jak zvuk, pa je približno delta. Njegova snimka je zato
**impulsni odziv prostorije**: izravni zvuk, rane refleksije od zidova i odjek koji polako
trne. Svaki uzorak suhog zapisa $x[k]$ pokrene u prostoriji kopiju odziva, umanjenu za
$x[k]$ i pomaknutu za $k$ uzoraka. Zbroj tih kopija je zvuk koji čujemo.

## 2. Učitavanje zapisa

Suhi zapis $x[n]$ snimljen je bez odjeka, a impulsni odziv $h[n]$ je snimka pucnja u
prostoriji. Zadani zapisi su **zamjenski**: generira ih skripta `alati/zamjenski_zvuk.py`.
Zamjenski suhi zapis ima dva kratka klika, četiri tona i završni akord, a zamjenski impulsni
odziv izravni zvuk, rane refleksije i odjek od oko 1 s. Izvor i licenca svakog zapisa
navedeni su u `LICENSE-sadrzaj.md`.

Vlastite zapise zadajte u *Postavkama* na vrhu (`ULAZ`, `ODZIV`). Funkcije u ćeliji ispod
pripreme svaki zapis i ispišu svaku promjenu:

- `ucitaj` čita WAV (8, 16, 24 i 32 bita), stereo pretvara u mono i po potrebi preuzorkuje;
  za datoteku koja ne postoji, nije WAV ili je tiha javi grešku s uputom;
- frekvenciju uzorkovanja $f_s$ određuje ulaz (najviše 48 kHz), a odziv se preuzorkuje na
  istu $f_s$ (npr. 48 kHz s mobitela na 44,1 kHz) jer konvolucija zbraja uzorke istog
  razmaka $T = 1/f_s$;
- ulazu se odreže dulja tišina na početku, a ulaz dulji od `MAX_ULAZ_S` skraćuje se;
- impulsnom odzivu odreže se tišina prije izravnog zvuka (prvi uzorak iznad 10 % vršne
  vrijednosti) jer bi inače sav izlaz kasnio. Vlastitom odzivu odreže se i tihi rep: dio
  ispod `REP_DB` (zadano −60 dB) ili ispod razine šuma snimke, koji bi samo dodao šum;
- oba zapisa normiraju se na vršnu vrijednost 0,9.

In [ ]:
FS_NAJVISE = 48000      # Hz: ulaz s višom f_s (npr. 96 kHz) preuzorkuje se na 48 kHz
MAX_ODZIV_S = 6         # s: dulji impulsni odziv skraćuje se (dvorane imaju RT60 do oko 3 s)
PRAG_TISINE = 0.03      # tišina na početku ulaza: ispod 3 % vršne vrijednosti (−30 dB)
PRAG_POCETKA = 0.1      # izravni zvuk odziva: prvi uzorak iznad 10 % vrha (−20 dB)
UPUTA_WAV = ("Bilježnica čita samo WAV. Pretvorite zapis u WAV, npr. u Audacityju: "
             "File → Open, zatim File → Export Audio… (vrsta WAV, Signed 16-bit PCM).")


def nadji(ime, mapa=ZVUK):
    """Putanja zapisa: ime s mapom (npr. "moj_zvuk/govor.wav") ili samo ime u mapi."""
    put = Path(ime).expanduser()
    kandidati = [put] if put.parent != Path(".") else [Path(mapa) / put, MOJ_ZVUK / put, put]
    for k in kandidati:
        if k.is_file():
            return k
    trazeno = ", ".join(k.as_posix() for k in kandidati)
    raise FileNotFoundError(f"Nema datoteke „{ime}” (traženo: {trazeno}). Putanja je "
                            "relativna prema mapi bilježnice, npr. \"moj_zvuk/govor.wav\".")


def preuzorkuj(s, fs_zapisa, fs, ime=""):
    """Preuzorkovanje s fs_zapisa na fs (polifazni filtar, scipy.signal.resample_poly)."""
    if fs_zapisa == fs:
        return s
    d = gcd(fs, fs_zapisa)
    print(f"{ime}: preuzorkovano s {fs_zapisa} Hz na {fs} Hz")
    return resample_poly(s, fs // d, fs_zapisa // d)


def ucitaj(ime, fs=None, mapa=ZVUK):
    """WAV -> (frekvencija uzorkovanja, mono signal u rasponu −1 … 1).

    Zadani ``fs``: zapis s drugom frekvencijom uzorkovanja preuzorkuje se na ``fs``."""
    put = nadji(ime, mapa)
    if put.suffix.lower() != ".wav":
        raise ValueError(f"{put.name}: nije WAV datoteka. {UPUTA_WAV}")
    try:
        with warnings.catch_warnings():        # dodatni podaci u WAV-u s mobitela
            warnings.simplefilter("ignore", wavfile.WavFileWarning)
            fs_zapisa, s = wavfile.read(put)
    except ValueError as e:                    # npr. MP3 preimenovan u .wav, komprimirani WAV
        raise ValueError(f"{put.name}: ne mogu pročitati kao WAV ({e}). {UPUTA_WAV}") from None
    if s.dtype == np.uint8:                    # 8-bitni WAV: 0 … 255, tišina = 128
        s = (s.astype(float) - 128) / 128
    elif np.issubdtype(s.dtype, np.integer):   # 16, 24 (učitan kao 32) i 32 bita
        s = s / np.iinfo(s.dtype).max
    s = np.asarray(s, dtype=float)
    if s.ndim == 2:                            # stereo -> mono
        s = s.mean(axis=1)
    if s.size == 0 or not np.max(np.abs(s)) > 1e-3:
        raise ValueError(f"{put.name}: zapis je prazan ili tih (vrh ispod −60 dB). "
                         "Provjerite snimku, npr. u Audacityju.")
    if fs is not None:
        s = preuzorkuj(s, fs_zapisa, fs, put.name)
        fs_zapisa = fs
    return fs_zapisa, s


def pripremi_ulaz(x, fs, max_s=15):
    """Bez duge tišine na početku, najviše max_s sekundi, vrh 0,9."""
    pocetak = int(np.flatnonzero(np.abs(x) > PRAG_TISINE * np.max(np.abs(x)))[0])
    if pocetak > int(0.5 * fs):                # ostavi 0,2 s tišine prije zvuka
        rez = pocetak - int(0.2 * fs)
        print(f"ulaz: odrezano {broj(rez / fs)} s tišine na početku")
        x = x[rez:]
    if len(x) > max_s * fs:
        print(f"ulaz: zapis traje {broj(len(x) / fs)} s; uzeto prvih {broj(max_s)} s "
              "(MAX_ULAZ_S u postavkama)")
        x = x[:int(max_s * fs)].copy()
        n = min(len(x), int(0.02 * fs))        # 20 ms mekog kraja: bez klika na rezu
        x[len(x) - n:] *= np.linspace(1, 0, n)
    return 0.9 * x / np.max(np.abs(x))


def pripremi_odziv(h, fs, rep_db=None, max_s=MAX_ODZIV_S):
    """Odziv od izravnog zvuka; rep ispod rep_db (i ispod šuma) odrezan; vrh 0,9."""
    pocetak = int(np.flatnonzero(np.abs(h) >= PRAG_POCETKA * np.max(np.abs(h)))[0])
    rez = max(0, pocetak - int(0.001 * fs))    # 1 ms prije izravnog zvuka ostaje
    tisina = h[:rez]                           # za procjenu šuma snimke
    if rez > 0:
        print(f"odziv: odrezano {broj(rez / fs)} s tišine prije izravnog zvuka")
        h = h[rez:]
    if len(h) > max_s * fs:
        print(f"odziv: traje {broj(len(h) / fs)} s; uzeto prvih {broj(max_s)} s")
        h = h[:int(max_s * fs)]
    if rep_db is not None:
        n = max(1, int(0.02 * fs))             # razina u kliznom prozoru od 20 ms
        snaga = np.maximum(fftconvolve(h**2, np.ones(n) / n, mode="same"), 0)
        razina = 10 * np.log10(snaga / snaga.max() + 1e-12)
        prag = rep_db
        if len(tisina) >= int(0.05 * fs):      # šum snimke iz tišine prije praska
            sum_db = 10 * np.log10(np.mean(tisina**2) / snaga.max() + 1e-12)
            prag = min(max(rep_db, sum_db + 6), -20)
        kraj = max(int(np.flatnonzero(razina > prag)[-1]) + 1, int(0.01 * fs))
        if kraj < len(h):
            h = h[:kraj].copy()
            m = min(kraj, int(0.01 * fs))      # 10 ms mekog kraja
            h[kraj - m:] *= np.linspace(1, 0, m)
            print(f"odziv: rep ispod {prag:.0f} dB odrezan; odziv traje "
                  f"{broj(len(h) / fs)} s".replace("-", "−"))
    return 0.9 * h / np.max(np.abs(h))

In [ ]:
# Ulaz određuje frekvenciju uzorkovanja f_s; odziv se preuzorkuje na istu f_s.
fs, x = ucitaj(ULAZ or "suhi_zvuk.wav")
if fs > FS_NAJVISE:
    x, fs = preuzorkuj(x, fs, FS_NAJVISE, "ulaz"), FS_NAJVISE
x = pripremi_ulaz(x, fs, MAX_ULAZ_S)
_, h = ucitaj(ODZIV or "odziv_pucanj.wav", fs=fs)
h = pripremi_odziv(h, fs, rep_db=REP_DB if ODZIV else None)   # zamjenski odziv je bez šuma
Tx, Th = len(x) / fs, len(h) / fs
print(f"frekvencija uzorkovanja f_s = {fs} Hz")
print(f"suhi zapis x:       N_x = {len(x)} uzoraka, T_x = {broj(Tx)} s")
print(f"impulsni odziv h:   N_h = {len(h)} uzoraka, T_h = {broj(Th)} s")

## 3. Slušanje: suhi zapis i pucanj u prostoriji

Najprije poslušajte oba zapisa. Suhi zapis zvuči „mrtvo”, kao u studiju. Impulsni odziv
počinje oštrim praskom (izravni zvuk), a zatim se čuje odjek koji trne.

In [ ]:
print("Suhi zapis x[n]:")
display(Audio(x, rate=fs))
print("Impulsni odziv h[n] (pucanj u prostoriji):")
display(Audio(h, rate=fs))

## 4. Konvolucija $y = x * h$

Izravni zbroj iz formule trebao bi oko $N_x \cdot N_h$ množenja (za zamjenske zapise oko
$9\cdot 10^9$; ćelija ispiše broj za učitane zapise).
Funkcija `scipy.signal.fftconvolve` daje isti rezultat preko brze Fourierove transformacije
(FFT, poglavlja 9 i 13) u djeliću sekunde. Rezultat normiramo na vršnu vrijednost 0,9 da
zvuk ne bude preglasan ni izobličen.

In [ ]:
y = fftconvolve(x, h)
assert len(y) == len(x) + len(h) - 1
y = 0.9 * y / np.max(np.abs(y))            # normiranje glasnoće
Ty = len(y) / fs
print(f"y: N_y = N_x + N_h − 1 = {len(x)} + {len(h)} − 1 = {len(y)} uzoraka")
print(f"trajanje: T_y = {broj(Ty)} s ≈ T_x + T_h = {broj(Tx)} s + {broj(Th)} s")
mantisa, eksponent = f"{len(x) * len(h):.1e}".split("e")
print(f"izravni zbroj iz formule: N_x · N_h ≈ {mantisa.replace('.', ',')}·10^{int(eksponent)} množenja")
print("Rezultat y[n], suhi zapis u prostoriji:")
display(Audio(y, rate=fs))

## 5. Tri signala na istoj vremenskoj osi

Sva tri signala crtamo na istoj vremenskoj osi. Izlaz $y$ traje dulje od ulaza $x$, i to
upravo za trajanje impulsnog odziva $T_h$: zadnji uzorak ulaza pokrene još jednu cijelu kopiju
odziva. Iza kraja ulaza (crtkana crta) izlaz ne prestaje naglo, nego trne (u zamjenskom zapisu
to je završni akord).

In [ ]:
fig, osi = plt.subplots(3, 1, figsize=(10, 6.5), sharex=True)
signali = [(x, BOJA_X, "$x[n]$", "suhi zapis $x[n]$"),
           (h, BOJA_H, "$h[n]$", "impulsni odziv $h[n]$ (pucanj u prostoriji)"),
           (y, BOJA_Y, "$y[n]$", "izlaz $y[n] = (x*h)[n]$")]
for ax, (s, boja, os_y, naslov) in zip(osi, signali):
    ax.plot(np.arange(len(s)) / fs, s, color=boja, lw=0.6)
    ax.set_ylabel(os_y)
    ax.set_title(naslov, loc="left", fontsize=10)
    ax.set_ylim(-1.05, 1.05)
    for granica, crta in ((Tx, "--"), (Ty, ":")):
        ax.axvline(granica, color="k", ls=crta, lw=1)
    ax.yaxis.set_major_formatter(zarez)
osi[2].annotate("", xy=(Ty, -0.8), xytext=(Tx, -0.8),
                arrowprops=dict(arrowstyle="<->", color="k"))
osi[2].text((Tx + Ty) / 2, -0.75, f"produljenje ≈ $T_h$ = {broj(Th)} s",
            ha="center", va="bottom")
osi[2].set_xlabel("vrijeme $t$ / s   (crtkano: kraj ulaza $T_x$; točkasto: kraj izlaza $T_y$)")
osi[2].set_xlim(0, Ty * 1.02)
osi[2].xaxis.set_major_formatter(zarez)
fig.tight_layout()
plt.show()

Na linearnom prikazu tihi kraj odjeka gotovo se ne vidi. Zato crtamo i **razinu u
decibelima**: srednju snagu u kliznom prozoru od 20 ms. Takav pomični prosjek i sam je konvolucija,
s pravokutnim impulsom (kao na predavanju). Razina izlaza nakon kraja ulaza pada
gotovo pravocrtno, za 60 dB u vremenu odjeka $RT_{60}$ (zamjenski odziv: $RT_{60} \approx 1$ s).
Odjek traje sve do kraja izlaza.

In [ ]:
def razina_db(s, prozor=0.02):
    """Razina u dB: srednja snaga u kliznom prozoru (s), 0 dB = najveća vrijednost."""
    n = int(prozor * fs)
    snaga = np.maximum(fftconvolve(s**2, np.ones(n) / n, mode="same"), 0)
    return 10 * np.log10(snaga / snaga.max() + 1e-12)


fig, ax = plt.subplots(figsize=(10, 3.2))
ax.plot(np.arange(len(x)) / fs, razina_db(x), color=BOJA_X, lw=1, label="ulaz $x$")
ax.plot(np.arange(len(y)) / fs, razina_db(y), color=BOJA_Y, lw=1, ls="--", label="izlaz $y$")
ax.axvline(Tx, color="k", ls="--", lw=1)
ax.axvline(Ty, color="k", ls=":", lw=1)
ax.axhline(-60, color=BOJA_H, lw=0.8)
ax.text(Ty - 0.05, -57, "−60 dB", ha="right", color=BOJA_H)
ax.set_ylim(-80, 3)
ax.set_xlim(0, Ty * 1.02)
ax.set_xlabel("vrijeme $t$ / s")
ax.set_ylabel("razina / dB")
ax.set_title("Razina u decibelima: odjek traje i nakon kraja ulaza", loc="left", fontsize=10)
ax.legend(loc="upper right")
ax.xaxis.set_major_formatter(zarez)
fig.tight_layout()
plt.show()

Pogledajmo pobliže početak impulsnog odziva. Prvi impuls je izravni zvuk, a nekoliko
idućih, unutar nekoliko desetaka milisekundi, su rane refleksije od zidova, poda i stropa.
Nakon njih refleksija je toliko da se stapaju u šum koji eksponencijalno trne: to je odjek.

In [ ]:
n = min(len(h), int(0.06 * fs))             # prvih 60 ms
fig, ax = plt.subplots(figsize=(10, 2.8))
ax.plot(1000 * np.arange(n) / fs, h[:n], color=BOJA_H, lw=0.8)
ax.set_xlabel("vrijeme $t$ / ms")
ax.set_ylabel("$h[n]$")
ax.set_title("Početak odziva: izravni zvuk ($t = 0$), rane refleksije, početak odjeka",
              loc="left", fontsize=10)
ax.yaxis.set_major_formatter(zarez)
fig.tight_layout()
plt.show()

## 6. Klizači: udio odjeka i duljina odziva

Klizač **udio odjeka** $a$ miješa suhi i „mokri” zvuk:
$y_a = (1-a)\,x + a\,(x * h_T)$. Za $a = 0$ čuje se samo suhi zapis, a za $a = 1$ samo
zvuk u prostoriji. Klizač **odziv / s** skraćuje impulsni odziv na prvih $T$ sekundi
($h_T$): kraći odziv znači kraći odjek, kao u manjoj ili prigušenijoj prostoriji, a izlaz je
dulji od ulaza samo za $T$. Nakon pomaka klizača bilježnica ponovno računa konvoluciju i nudi
novi zvuk za puštanje.

Ako se klizači ne prikažu, pozovite funkciju izravno, npr. `odjek(udio=0.5, odziv_s=0.3)`.

In [ ]:
def odjek(udio=0.7, odziv_s=None):
    """Mješavina suhog i mokrog zvuka; odziv skraćen na odziv_s sekundi (None: cijeli)."""
    n_h = len(h) if odziv_s is None else min(len(h), max(2, int(odziv_s * fs)))
    h_T = h[:n_h].copy()
    n_rub = min(n_h, int(0.01 * fs))            # 10 ms mekog kraja: bez klika na rezu
    h_T[n_h - n_rub:] *= np.linspace(1, 0, n_rub)
    mokri = fftconvolve(x, h_T)
    mokri /= np.max(np.abs(mokri))
    suhi = np.pad(x / np.max(np.abs(x)), (0, len(mokri) - len(x)))
    y_a = (1 - udio) * suhi + udio * mokri
    y_a = 0.9 * y_a / np.max(np.abs(y_a))
    fig, ax = plt.subplots(figsize=(10, 2.5))
    ax.plot(np.arange(len(y_a)) / fs, y_a, color=BOJA_Y, lw=0.6)
    ax.axvline(Tx, color="k", ls="--", lw=1)
    ax.set_xlim(0, Tx + Th)
    ax.set_ylim(-1.05, 1.05)
    ax.set_xlabel("vrijeme $t$ / s")
    ax.set_title(f"udio odjeka {broj(udio)}, odziv {broj(n_h / fs)} s: "
                  f"izlaz traje {broj(len(y_a) / fs)} s", loc="left", fontsize=10)
    ax.xaxis.set_major_formatter(zarez)
    ax.yaxis.set_major_formatter(zarez)
    fig.tight_layout()
    plt.show()
    display(Audio(y_a, rate=fs))


# Klizač duljine odziva ide do T_h (korak 0,05 s; za odziv kraći od 0,5 s korak 0,01 s).
korak = 0.05 if Th >= 0.5 else 0.01
najdulji = round(max(korak, np.ceil(Th / korak - 1e-9) * korak), 2)
klizaci = widgets.interactive(
    odjek,
    udio=widgets.FloatSlider(value=0.7, min=0, max=1, step=0.05, description="udio odjeka",
                             continuous_update=False),
    odziv_s=widgets.FloatSlider(value=najdulji, min=korak, max=najdulji, step=korak,
                                description="odziv / s", continuous_update=False),
)
display(klizaci)

## 7. Veza s formulom $y = x * h$

- **Zbroj pomaknutih kopija.** Svaki uzorak $x[k]$ pokrene kopiju odziva $x[k]\,h[n-k]$, a
  izlaz je njihov zbroj: to je upravo formula $y[n]=\sum_k x[k]\,h[n-k]$. Ćelija ispod
  provjerava da `fftconvolve` daje isto što i zbroj iz formule.
- **Klik je gotovo delta.** Nakon klika čuje se sam odjek prostorije jer je
  $\delta * h = h$. Delta je jedinica konvolucije: $x * \delta = x$.
- **Produljenje.** $N_y = N_x + N_h - 1$, odnosno $T_y \approx T_x + T_h$, kao u grafičkoj
  konvoluciji dvaju pravokutnih impulsa (trajanja 1 s i 2 s daju rezultat od 3 s).
- **Jeka s predavanja.** Odziv $h(t)=\delta(t)+\tfrac{1}{2}\,\delta(t-2)$ ima dva impulsa
  i daje jednu jeku. Prostorija ima tisuće refleksija, koje se stapaju u odjek.
- **Razmazani tonovi.** Ton još zvuči dok počinje idući, pa se tonovi preklapaju. Zato se
  govor u velikoj dvorani s dugim odjekom teže razumije.

Za čitanje: udžbenik *Signali i sustavi*, **poglavlje 8, Konvolucija i impulsni odziv**
(grafička konvolucija, svojstva konvolucije i primjer jeke), i predavanje P08.

In [ ]:
# Provjera 1: fftconvolve daje isto što i zbroj iz formule (kratki isječci signala).
# Isječak x počinje na prvom glasnom mjestu (u zamjenskom zapisu na prvom kliku): zapis
# do njega šuti, a s nulama provjera ne bi ništa dokazala. Odziv počinje izravnim zvukom.
p = int(np.flatnonzero(np.abs(x) > 0.3 * np.max(np.abs(x)))[0])
xk, hk = x[p:p + 300], h[:200]
assert np.max(np.abs(xk)) > 0.05 and np.max(np.abs(hk)) > 0.05, "isječak šuti"
y_formula = np.array([sum(xk[k] * hk[n - k] for k in range(len(xk)) if 0 <= n - k < len(hk))
                      for n in range(len(xk) + len(hk) - 1)])
razlika = np.max(np.abs(fftconvolve(xk, hk) - y_formula))
print(f"najveća razlika fftconvolve − zbroj iz formule: {razlika:.1e}")
assert razlika < 1e-9

# Provjera 2: delta je jedinica konvolucije, a pomaknuta delta kasni signal
delta = np.zeros(fs // 2)
delta[0] = 1
assert np.allclose(fftconvolve(x, delta)[:len(x)], x)
kasnjenje = fs // 4                          # 0,25 s
delta_T = np.zeros(fs // 2)
delta_T[kasnjenje] = 1
assert np.allclose(fftconvolve(x, delta_T)[kasnjenje:kasnjenje + len(x)], x)
print("x * δ = x i x * δ(t − 0,25 s) = x(t − 0,25 s): provjereno")

## Pokušajte sami

1. Postavite udio odjeka na 1 i skraćujte odziv od pune duljine (1,5 s za zamjenski odziv)
   prema 0,05 s. Kada odjek prestaje biti čujan, a kada se još čuju samo rane refleksije?
2. Napravite odziv s drugim vremenom odjeka u zasebnoj mapi (zamjenski zapisi ostaju
   nepromijenjeni): `python alati/zamjenski_zvuk.py moj_zvuk --rt60 2.5` (ili `--rt60 0.3`).
   U postavkama na vrhu upišite `ODZIV = "moj_zvuk/odziv_pucanj.wav"` i pokrenite sve ćelije.
   Rep ispod −60 dB bilježnica odreže, pa odziv traje oko RT60 i klizač duljine to prati.
3. Snimite pljesak ili prasak balona u hodniku ili stubištu (upute u *Postavkama*), spremite
   ga kao WAV u `moj_zvuk/` (nije u gitu) i upišite ga kao `ODZIV`. Kao `ULAZ` upišite
   vlastitu snimku govora. Zašto se govor u prostoriji s dugim odjekom teže razumije?
4. Izračunajte i poslušajte $h * h$. Koju prostoriju (ili spoj dviju prostorija) opisuje?
   Uputa: asocijativnost konvolucije, $(x * h) * h = x * (h * h)$.

---
Zvučni zapisi: vlastiti zamjenski zapisi, izvor i licenca u `LICENSE-sadrzaj.md`. Kod je pod
licencom MIT (`LICENSE`), a tekst pod CC BY 4.0.